# Clase Abstracta: La Interfaz `IList`

En el diseño de estructuras de datos orientado a objetos, es fundamental separar **qué** hace una estructura de **cómo** lo hace. 

Para lograr esto, utilizamos una **Clase Abstracta** (o Interfaz). Esta clase define un "contrato" que cualquier tipo de lista (Simple, Doblemente Enlazada, Circular, Vector) debe cumplir. Si una clase quiere ser considerada una "Lista", debe implementar estos métodos.

## Código de la Interfaz

A continuación se presenta la definición completa de la interfaz `IList` en C++.

```cpp
#ifndef ILIST_H
#define ILIST_H

#include <cstddef> // Necesario para el tipo size_t (enteros sin signo)

/**
 * Interfaz abstracta para listas (Simple, Doble, Circular).
 * Define el contrato que todas las listas deben cumplir.
 */
template <typename T>
class IList {
public:
    // Destructor virtual: CRUCIAL para el polimorfismo.
    // Garantiza que se llame al destructor de la clase derivada al hacer delete.
    virtual ~IList() {}

    // --- Métodos de Inserción ---
    
    // Inserta un elemento al final de la lista
    virtual void pushBack(const T& item) = 0;

    // Inserta un elemento al inicio de la lista
    virtual void pushFront(const T& item) = 0;

    // Inserta un elemento en una posición específica
    virtual void insertAt(size_t index, const T& item) = 0;

    // --- Métodos de Eliminación ---

    // Elimina el elemento en una posición específica
    virtual void removeAt(size_t index) = 0;

    // Elimina el primer elemento
    virtual void popFront() = 0;

    // Elimina el último elemento
    virtual void popBack() = 0;
    
    // Limpia toda la lista
    virtual void clear() = 0;

    // --- Métodos de Acceso y Estado ---

    // Devuelve el elemento en un índice dado (solo lectura)
    virtual T get(size_t index) const = 0;

    // Devuelve el tamaño actual de la lista
    virtual size_t getSize() const = 0;

    // Verifica si la lista está vacía
    virtual bool isEmpty() const = 0;

    // Imprime la lista (útil para depuración)
    virtual void print() const = 0;
};

#endif // ILIST_H

## Análisis Detallado de los Componentes
1. Guardas de Inclusión (#ifndef)
   Estas líneas previenen errores de compilación si el archivo IList.h es incluido múltiples veces en el mismo proyecto. Es un estándar en C/C++.
```cpp
#ifndef ILIST_H
#define ILIST_H
...
#endif



2. Templates (Plantillas)
Esto permite que nuestra lista sea genérica. No creamos una lista solo para enteros (int) o solo para texto (string). Con T, el usuario define el tipo de dato al momento de crear la lista:

```cpp
template <typename T>
IList<int>
IList<float>
IList<Usuario>



3. Funciones Virtuales Puras
La sintaxis `virtual ... = 0`; indica que el método es virtual puro.

* Esta clase IList no tiene la implementación del método (no sabe cómo insertar o borrar).
* No se pueden crear objetos de tipo IList directamente (IList x; dará error).
* Obliga a cualquier clase hija (como DLList) a escribir el código para cada una de estas funciones. Si falta una, la hija también será abstracta y no compilará.

4. El Destructor Virtual
Este es quizás el punto más importante para evitar fugas de memoria (Memory Leaks).

* **Sin virtual**: El compilador solo llamaría al destructor de IList, ignorando el de DLList, dejando los nodos en memoria.
* **Con virtual**: El programa sabe que debe buscar el destructor de la clase derivada (DLList) primero, limpiando correctamente la memoria.
Si tenemos el siguiente escenario:
```cpp
virtual ~IList() {}

IList<int>* miLista = new CircularList<int>();
// ... uso de la lista ...
delete miLista; // Aquí ocurre la magia

## ¿Por qué usar esta Interfaz?
El uso de `IList` nos permite aplicar el principio de Polimorfismo. Podemos escribir funciones que reciban una lista genérica y funcionen con cualquier implementación futura.

Ejemplo de Polimorfismo:

```cpp
// Esta función funciona con CUALQUIER lista que herede de IList
// No le importa si es circular, doble o simple.
void llenarLista(IList<int>* lista) {
    lista->pushBack(10);
    lista->pushBack(20);
    lista->pushBack(30);
    lista->print();
}

# Lista Doblemente Enlazada Circular (DLList)

Esta implementación de DLList es una estructura de datos lineal avanzada que combina eficiencia y seguridad mediante el uso de Nodos Centinela (Dummy), Circularidad y Semántica de Movimiento (Move Semantics) de C++ moderno.

A continuación, se detalla la anatomía de la clase paso a paso.

## Estructura Fundamental: El Nodo (DNode)

El bloque constructivo de la lista es la estructura DNode. A diferencia de una lista simple, este nodo mantiene dos punteros.

```cpp
template <typename T>
struct DNode {
    T data;       // El dato almacenado
    DNode* prev;  // Puntero al nodo anterior
    DNode* next;  // Puntero al nodo siguiente
    // ...
};


### Características de los Constructores del Nodo:

* **Constructor por Defecto**: `DNode()`. Inicializa punteros a nullptr. Se usa exclusivamente para crear el nodo dummy.
* **Constructor de Copia**: `DNode(const T& val)`. Copia el dato por valor. Es el estándar para tipos de datos primitivos (int, double).
* **Constructor de Movimiento**: `DNode(T&& val)`. Utiliza std::move para transferir la propiedad del recurso. Es crítico cuando T es un objeto pesado (ej. una Imagen o una Base de Datos), evitando copias costosas.

### Arquitectura de la Clase DLList

* La clase hereda de `IList<T>`, asegurando polimorfismo.
* El Patrón del Nodo Centinela (dummy)
  En lugar de gestionar punteros head y tail que pueden ser nullptr, utilizamos un único nodo ficticio (dummy).
* Estado Vacío: En una lista vacía, dummy->next y dummy->prev apuntan al mismo dummy.

### Circularidad:

* El "primer" elemento real es dummy->next.
* El "último" elemento real es dummy->prev.
* Ventaja Técnica: Elimina la necesidad de condicionales if (head == nullptr) en las operaciones de inserción y borrado.

### Optimización de Búsqueda (getNode)

El método privado getNode(index) implementa una búsqueda inteligente:
* Si el índice está en la primera mitad, itera hacia adelante desde el inicio.
* Si el índice está en la segunda mitad, itera hacia atrás desde el final.
* Impacto: Reduce el tiempo promedio de búsqueda a la mitad ($n/2$), aunque la complejidad asintótica permanece $O(n)$.


## El Iterador Bidireccional (class Iterator)

Permite recorrer la lista abstractrayendo la estructura de punteros. Cumple con los requisitos de `std::bidirectional_iterator_tag`.

### Operadores Implementados

* operator*: Acceso al dato (curr->data).
* operator++ (Pre/Post): Avanza al siguiente nodo (`curr = curr->next`). En una lista circular, avanzar desde el último elemento nos lleva al dummy (que representa `end()`).
* operator--: Retrocede al nodo anterior. Retroceder desde `end()` (`dummy`) nos lleva al último elemento real.

```cpp
#ifndef DLLIST_H
#define DLLIST_H

#include "ilist.h"
#include <stdexcept>
#include <iostream>
#include <utility> // Necesario para std::exchange y std::move

// --- Estructura del Nodo ---
template <typename T>
struct DNode {
    T data;
    DNode* prev;
    DNode* next;

    // Constructor por defecto (para el Dummy)
    DNode() : prev(nullptr), next(nullptr) {}

    // Constructor copia
    DNode(const T& val) : data(val), prev(nullptr), next(nullptr) {}
    
    // Constructor movimiento (Opcional, pero recomendado para T complejos)
    DNode(T&& val) : data(std::move(val)), prev(nullptr), next(nullptr) {}
};

/**
 * Lista Doblemente Enlazada Circular con Nodo Centinela (Dummy).
 * Implementa IList y Semántica de Movimiento completa.
 */
template <typename T>
class DLList : public IList<T> {
private:
    DNode<T>* dummy; // Nodo centinela
    size_t size;

    // Método auxiliar para localizar nodos
    DNode<T>* getNode(size_t index) const {
        if (index < 0 || index >= size) return nullptr;

        DNode<T>* curr;
        if (index < size / 2) {
            curr = dummy->next;
            for (size_t i = 0; i < index; ++i) curr = curr->next;
        } else {
            curr = dummy->prev;
            for (size_t i = size - 1; i > index; --i) curr = curr->prev;
        }
        return curr;
    }

    void init() {
        dummy = new DNode<T>();
        dummy->next = dummy;
        dummy->prev = dummy;
        size = 0;
    }

public:
    // ==========================================
    // 1. Constructores y Destructores
    // ==========================================

    DLList() { init(); }

    virtual ~DLList() {
        // Importante verificar dummy por si el objeto fue movido (moved-from state)
        if (dummy) {
            clear();
            delete dummy;
        }
    }

    // --- Constructor de Copia (Deep Copy) ---
    DLList(const DLList& other) {
        init();
        for (auto it = other.begin(); it != other.end(); ++it) {
            pushBack(*it);
        }
    }

    // Constructor por Movimiento (Move Constructor)
    // Usa std::exchange para "robar" los punteros y dejar al otro en nullptr/0
    DLList(DLList&& other) noexcept 
        : dummy(std::exchange(other.dummy, nullptr)), 
          size(std::exchange(other.size, 0)) {
    }

    // Operador de Asignación por Movimiento (Move Assignment)
    DLList& operator=(DLList&& other) noexcept {
        if (this != &other) {
            // 1. Liberar recursos propios
            if (dummy) {
                clear();
                delete dummy;
            }

            // 2. Robar recursos del otro
            dummy = std::exchange(other.dummy, nullptr);
            size = std::exchange(other.size, 0);
        }
        return *this;
    }

    // ==========================================
    // 2. Iteradores Bidireccionales
    // ==========================================
    
    class Iterator {
    private:
        DNode<T>* curr;
    public:
        // Rasgos (Traits) para compatibilidad con STL
        using iterator_category = std::bidirectional_iterator_tag;
        using value_type = T;
        using difference_type = std::ptrdiff_t;
        using pointer = T*;
        using reference = T&;

        Iterator(DNode<T>* ptr) : curr(ptr) {}

        T& operator*() { return curr->data; }
        DNode<T>* getNode() { return curr; } 

        Iterator& operator++() { // ++it
            curr = curr->next;
            return *this;
        }

        Iterator operator++(int) { // it++
            Iterator temp = *this;
            curr = curr->next;
            return temp;
        }

        Iterator& operator--() { // --it
            curr = curr->prev;
            return *this;
        }

        bool operator!=(const Iterator& other) const { return curr != other.curr; }
        bool operator==(const Iterator& other) const { return curr == other.curr; }
    };

    Iterator begin() const { return Iterator(dummy ? dummy->next : nullptr); }
    Iterator end() const { return Iterator(dummy); }

    // ==========================================
    // 3. Implementación de IList
    // ==========================================

    void pushBack(const T& item) override {
        if (!dummy) init(); 
        DNode<T>* newNode = new DNode<T>(item);
        DNode<T>* tail = dummy->prev; 

        newNode->next = dummy;
        newNode->prev = tail;
        tail->next = newNode;
        dummy->prev = newNode;
        size++;
    }

    void pushFront(const T& item) override {
        if (!dummy) init();
        DNode<T>* newNode = new DNode<T>(item);
        DNode<T>* head = dummy->next;

        newNode->prev = dummy;
        newNode->next = head;
        head->prev = newNode;
        dummy->next = newNode;
        size++;
    }

    void insertAt(size_t index, const T& item) override {
        if (index > size) throw std::out_of_range("Indice fuera de rango"); // index puede ser igual a size
        if (index == size) { pushBack(item); return; }
        if (index == 0) { pushFront(item); return; }

        DNode<T>* current = getNode(index);
        DNode<T>* prevNode = current->prev;
        DNode<T>* newNode = new DNode<T>(item);

        newNode->prev = prevNode;
        newNode->next = current;
        prevNode->next = newNode;
        current->prev = newNode;
        size++;
    }

    void removeAt(size_t index) override {
        if (index >= size) throw std::out_of_range("Indice fuera de rango");
        DNode<T>* toDelete = getNode(index);
        
        toDelete->prev->next = toDelete->next;
        toDelete->next->prev = toDelete->prev;
        delete toDelete;
        size--;
    }

    void popFront() override { if (!isEmpty()) removeAt(0); }
    void popBack() override { if (!isEmpty()) removeAt(size - 1); }

    void clear() override {
        if (!dummy) return;
        while (size > 0) popFront();
    }

    T get(size_t index) const override {
        DNode<T>* node = getNode(index);
        if (!node) throw std::out_of_range("Indice fuera de rango");
        return node->data;
    }

    size_t getSize() const override { return size; }
    bool isEmpty() const override { return size == 0; }

    void print() const override {
        if (!dummy || size == 0) {
            std::cout << "[]" << std::endl;
            return;
        }
        DNode<T>* curr = dummy->next;
        std::cout << "[ ";
        while (curr != dummy) {
            std::cout << curr->data << " ";
            curr = curr->next;
        }
        std::cout << "]" << std::endl;
    }
};

#endif // DLLIST_H

# Ejemplo de uso

In [1]:
#include <iostream>
#include <string>
#include "dllist.h"

void test(){
    DLList<std::string> superHeroes;

    std::cout << "--- 1. Probando Inserciones ---" << std::endl;
    superHeroes.pushBack("Iron Man");
    superHeroes.pushBack("Spider-Man");
    superHeroes.pushFront("Thor");
    superHeroes.insertAt(1, "Hulk"); // Thor, Hulk, Iron Man, Spider-Man
    superHeroes.print();

    std::cout << "\n--- 2. Probando Iteradores (C++11) ---" << std::endl;
    for (const auto& hero : superHeroes) {
        std::cout << "Vengador: " << hero << std::endl;
    }

    std::cout << "\n--- 3. Probando Eliminaciones ---" << std::endl;
    superHeroes.popFront(); // Quita a Thor
    superHeroes.popBack();  // Quita a Spider-Man
    superHeroes.print();

    std::cout << "\n--- 4. Estado de la lista ---" << std::endl;
    std::cout << "Tamano final: " << superHeroes.getSize() << std::endl;
    std::cout << "Elemento en indice 1: " << superHeroes.get(1) << std::endl;
}

test();

--- 1. Probando Inserciones ---
[ Thor Hulk Iron Man Spider-Man ]

--- 2. Probando Iteradores (C++11) ---
Vengador: Thor
Vengador: Hulk
Vengador: Iron Man
Vengador: Spider-Man

--- 3. Probando Eliminaciones ---
[ Hulk Iron Man ]

--- 4. Estado de la lista ---
Tamano final: 2
Elemento en indice 1: Iron Man
